[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_02/4_ai.ipynb)

# Day 2: AI and ML (easy)

**How to work:** Answer concept questions out loud in 60 to 90 seconds before opening the solution: definition, why it matters, one example, one trade-off.

**Today (24 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Two kinds of error | easy | 4 |
| Q2 | How many neighbours should vote? | easy | 7 |
| Q3 | A tree that memorises | easy | 6 |
| Q4 | Read the learning numbers | easy | 4 |
| Q5 | A random split for next month's sales (review) | easy | 3 |

**Handbook:** [AI and ML](https://mahsa-agz.github.io/ds-handbook/#ai), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Data

- `mpg`: 398 cars (1970 to 1982). `mpg`, `cylinders`, `displacement`, `horsepower` (has NaN), `weight`, `acceleration`, `model_year`, `origin`, `name`.
- `load_breast_cancer()` from scikit-learn (built in, no download): 569 tumours, 30 numeric features, target 1 = benign, 0 = malignant.

The setup cell loads every table into a pandas DataFrame (from your Drive folder, or downloads it). Everything runs on the free Colab CPU in seconds. Numbers in the solutions come from scikit-learn 1.8; other versions can differ in the last digit.

## Setup

Run this cell once before the questions.

In [ ]:
# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

import io, zipfile, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.width", 120)

SEABORN = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/"
TELCO_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
BANK_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"

def load_telco():
    """IBM Telco churn, 7,043 customers. Target: Churn (Yes/No)."""
    return pd.read_csv(data_path("telco_churn.csv", TELCO_URL))

def load_seaborn(name):
    """titanic, penguins, diamonds, mpg, tips ... from the seaborn-data repo."""
    return pd.read_csv(data_path(name + ".csv", SEABORN + name + ".csv"))

def load_bank():
    """UCI Bank Marketing (bank-additional-full), 41,188 calls. Target: y (yes/no). The source is a zip in a zip."""
    path = data_path("bank_marketing.csv")
    if not os.path.exists(path):
        print("downloading bank_marketing (zip)")
        outer = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(BANK_URL).read()))
        inner = zipfile.ZipFile(io.BytesIO(outer.read("bank-additional.zip")))
        with open(path, "wb") as f:
            f.write(inner.read("bank-additional/bank-additional-full.csv"))
    return pd.read_csv(path, sep=";")

def check(fn, cases, tol=1e-6):
    """Runs fn on each (args, expected) case; numbers and arrays are compared with a tolerance."""
    ok = 0
    for args, want in cases:
        args = args if isinstance(args, tuple) else (args,)
        try:
            got = fn(*args)
            good = got is not None and np.shape(got) == np.shape(want) and bool(
                np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), atol=tol))
        except Exception as e:
            got, good = "error: " + repr(e)[:120], False
        ok += good
        shown = " ".join(repr(got).split())
        shown = shown if len(shown) < 100 else shown[:97] + "..."
        print(("PASS " if good else "FAIL ") + f"{fn.__name__}(...) -> {shown}" + ("" if good else f"   expected {want!r}"))
    print(f"{ok}/{len(cases)} passed")

mpg = load_seaborn("mpg")
print("mpg", mpg.shape)

---
### Q1. Two kinds of error

*easy, about 4 min*

Explain the bias-variance trade-off in about 60 seconds. Include the decomposition of the expected squared error, one example of a high-bias model and one of a high-variance model, and how model complexity moves each term.

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: 'why does my test error go up when the model gets more flexible?'. Pattern: bias-variance decomposition.

</details>

<details><summary><b>Hint 2</b></summary>

1. Write `expected error = bias^2 + variance + irreducible noise`. 2. Bias: error from wrong assumptions (too simple). 3. Variance: sensitivity to the particular training sample (too flexible). 4. Complexity up: bias down, variance up, so test error is U-shaped.

</details>

<details><summary><b>Solution</b></summary>

**Model answer (about 60 s):**

"For squared loss, the expected test error at a point splits into three parts: `E[(y - f_hat(x))^2] = bias^2 + variance + noise`. Bias is how far the average prediction, over many possible training sets, is from the truth: it comes from a model that is too simple, like a straight line through a curved relationship. Variance is how much the prediction changes when the training set changes: a deep decision tree or a degree-15 polynomial fits the noise, so a new sample gives a very different model. Noise is the part no model can remove.

As complexity grows, bias falls and variance rises, so the test error is U-shaped and training error keeps falling. We pick complexity with validation data. To cut variance: more data, regularization, simpler models, bagging. To cut bias: richer features, a more flexible model, less regularization, boosting."

**Key formula:** `E[(y - f_hat)^2] = bias^2 + variance + sigma^2`.

**Why:** This is the most common ML warm-up question. A strong answer gives the formula, defines both terms over repeated training sets (not over test points), and ends with the levers for each side.

**Common mistakes:** Saying 'bias = training error, variance = test error' (close as a diagnostic, wrong as a definition). Forgetting the noise term. Claiming more data reduces bias (it mainly reduces variance).

**Learn more:** [ai-bias-variance](https://mahsa-agz.github.io/ds-handbook/#ai-bias-variance)

</details>

---
### Q2. How many neighbours should vote?

*easy, about 7 min*

Use `mpg` (drop rows where `horsepower` is missing). Predict `mpg` from `horsepower` and `weight` with k-nearest-neighbours regression, for k = 1, 3, 10, 30, 100 and 300.

For each k, print the mean **train** RMSE and the mean **validation** RMSE over 5-fold CV (`KFold(5, shuffle=True, random_state=0)`). Scale the two features first.

Which k would you pick? Where do you see high variance and where high bias? Is a small k a simple or a complex model?

In [ ]:
# mpg is loaded. Your code here

<details><summary><b>Hint 1</b></summary>

Signal: one knob for model flexibility and a question about train versus validation error. Pattern: bias-variance curve with cross-validation.

</details>

<details><summary><b>Hint 2</b></summary>

1. `make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))`. 2. `cross_validate(..., scoring='neg_root_mean_squared_error', return_train_score=True)`. 3. Flip the sign of the scores and average them. 4. Pick the k with the lowest validation RMSE.

</details>

<details><summary><b>Solution</b></summary>

```python
from sklearn.model_selection import KFold, cross_validate
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d = mpg.dropna(subset=["horsepower"])
X, y = d[["horsepower", "weight"]], d["mpg"]
cv = KFold(5, shuffle=True, random_state=0)
for k in [1, 3, 10, 30, 100, 300]:
    model = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))
    r = cross_validate(model, X, y, cv=cv, scoring="neg_root_mean_squared_error", return_train_score=True)
    print(f"k {k:3d}: train RMSE {-r['train_score'].mean():.2f}  validation RMSE {-r['test_score'].mean():.2f}")
# out: k   1: train RMSE 0.61  validation RMSE 5.57
# out: k   3: train RMSE 3.14  validation RMSE 4.50
# out: k  10: train RMSE 3.66  validation RMSE 4.03
# out: k  30: train RMSE 3.78  validation RMSE 3.86
# out: k 100: train RMSE 4.11  validation RMSE 4.15
# out: k 300: train RMSE 7.52  validation RMSE 7.54
```

**Why:** Small k is the most flexible model: with k = 1 each training car predicts itself (train RMSE 0.61) but validation RMSE is 5.57, a huge gap, so high variance. Large k averages almost the whole data: with k = 300 train and validation are both bad (7.52 and 7.54), so high bias. The validation curve is U-shaped with its minimum at k = 30 (3.86), where the train/validation gap is small. Pick k = 30 (or search 20 to 50 more finely).

**Complexity:** kNN has no training cost; each prediction scans the training set, `O(n * p)`. Here well under a second.

**Common mistakes:** Choosing k by train error (k = 1 always wins). Not scaling: weight is in thousands of pounds and would dominate the distance. Forgetting that sklearn returns negative RMSE.

**Learn more:** [ai-bias-variance](https://mahsa-agz.github.io/ds-handbook/#ai-bias-variance), [ai-regularization-cv](https://mahsa-agz.github.io/ds-handbook/#ai-regularization-cv)

</details>

---
### Q3. A tree that memorises

*easy, about 6 min*

Use scikit-learn's breast cancer data. Split 70/30 (stratified, `random_state=0`). Fit `DecisionTreeClassifier(max_depth=d, random_state=0)` for `d` in 1, 2, 3, 5, 8 and `None` (no limit), and print train and test accuracy for each.

Which depths are high bias, which are high variance? What does the train accuracy of the unlimited tree tell you?

In [ ]:
from sklearn.datasets import load_breast_cancer

# your code here

<details><summary><b>Hint 1</b></summary>

Signal: a single complexity knob (depth) and a train/test gap. Pattern: diagnose bias versus variance from the two errors.

</details>

<details><summary><b>Hint 2</b></summary>

1. `load_breast_cancer(return_X_y=True)`. 2. Loop over depths, fit, use `.score` on train and test. 3. Small gap but low scores = bias; train near 1.0 with a lower test score = variance.

</details>

<details><summary><b>Solution</b></summary>

```python
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
for depth in [1, 2, 3, 5, 8, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_tr, y_tr)
    print(f"max_depth {str(depth):>4}: train {tree.score(X_tr, y_tr):.3f}  test {tree.score(X_te, y_te):.3f}")
# out: max_depth    1: train 0.932  test 0.889
# out: max_depth    2: train 0.942  test 0.906
# out: max_depth    3: train 0.980  test 0.901
# out: max_depth    5: train 0.997  test 0.912
# out: max_depth    8: train 1.000  test 0.906
# out: max_depth None: train 1.000  test 0.906
```

**Why:** Depth 1 and 2 are high bias: train accuracy is only 0.932 and 0.942. From depth 8 on, the tree is pure on the training data (train 1.000) but test stays at 0.906: the extra splits memorise single tumours, which is variance. Train accuracy 1.000 for the unlimited tree says nothing about quality; it only shows the tree can memorise. The best test score here is depth 5 (0.912), but all depths from 2 up sit within about 1 point, so prefer the simpler tree (or tune depth with CV).

**Complexity:** Each tree is `O(n * p * log n)` to grow; here a few milliseconds.

**Common mistakes:** Reading one test split as the truth: with 171 test rows, one tumour is 0.6 points, so differences of 1 to 2 points between depths are within noise. Use CV to choose the depth.

**Learn more:** [ai-bias-variance](https://mahsa-agz.github.io/ds-handbook/#ai-bias-variance), [ai-trees-boosting](https://mahsa-agz.github.io/ds-handbook/#ai-trees-boosting)

</details>

---
### Q4. Read the learning numbers

*easy, about 4 min*

Three teams report their model errors (classification error rate). The best achievable error (human level) is about 1%.

- Team A: train 14%, validation 15%.
- Team B: train 1%, validation 12%.
- Team C: train 7%, validation 18%.

Diagnose each and give two concrete next steps per team.

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: train and validation errors next to a target error. Pattern: bias = gap from train to the best possible error; variance = gap from train to validation.

</details>

<details><summary><b>Hint 2</b></summary>

1. Avoidable bias = train error minus 1%. 2. Variance = validation minus train. 3. Fix the larger gap first.

</details>

<details><summary><b>Solution</b></summary>

| Team | Avoidable bias (train - 1%) | Variance (val - train) | Diagnosis |
|---|---|---|---|
| A | 13 points | 1 point | High bias (underfitting) |
| B | 0 points | 11 points | High variance (overfitting) |
| C | 6 points | 11 points | Both |

- **A:** a more flexible model (trees or boosting instead of a linear model, more depth), better features, less regularization. More data will not help much: train and validation already agree.
- **B:** more training data or data augmentation, stronger regularization (L2, dropout, smaller depth, min samples per leaf), early stopping, bagging, fewer features.
- **C:** fix variance and bias together: better features plus regularization; also check that train and validation come from the same distribution (an 11-point gap can be data mismatch, not only overfitting).

**Why:** The two gaps tell you which lever to pull. The examiner wants you to compare train error with the best achievable error, not just with zero.

**Common mistakes:** Calling Team A 'fine' because the gap is small. Recommending more data for a high-bias model. Ignoring a train/validation distribution mismatch.

**Learn more:** [ai-bias-variance](https://mahsa-agz.github.io/ds-handbook/#ai-bias-variance)

</details>

---
### Q5. A random split for next month's sales (review)

*easy, about 3 min*

A colleague forecasts next month's daily sales. She uses a random 80/20 split of three years of daily rows, with features like 'sales 7 days ago' and 'mean sales of this month'. Validation MAPE is 4%; in production it is 15%. What went wrong and how should she validate?

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: time-ordered data and a big offline/online gap. Pattern: leakage through the split (and through features computed with future data).

</details>

<details><summary><b>Hint 2</b></summary>

1. Random split puts future days in train. 2. 'Mean sales of this month' uses days after the prediction date. 3. Use a time-based split that mimics production.

</details>

<details><summary><b>Solution</b></summary>

**Model answer:** "Two leaks. First, a random split mixes days: the model trains on, say, the 14th and 16th and is tested on the 15th, so it interpolates instead of forecasting. Second, 'mean sales of this month' is computed with the whole month, including days after the prediction date, so it contains the answer. In production neither is available, so error jumps from 4% to 15%.

Fix: build features only from data before the forecast date (lags of at least the forecast horizon, rolling means that end at the cutoff), and validate with a time split: train on months 1 to k, test on month k+1, then roll forward (expanding window, `TimeSeriesSplit`). Report the average over several cutoffs."

**Why:** Leakage through the split is the most common reason an offline score does not hold online. The rule: validation must copy the production situation, including what is known at prediction time.

**Common mistakes:** Only fixing the split and keeping the leaky feature. Using lag-1 features for a 30-day-ahead forecast.

**Learn more:** [ai-ml-framing](https://mahsa-agz.github.io/ds-handbook/#ai-ml-framing), [ai-regularization-cv](https://mahsa-agz.github.io/ds-handbook/#ai-regularization-cv)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Done for today.** Finish with the [theory flashcards for day 2](https://mahsa-agz.github.io/ds-handbook/#day-2).